## Fetching entire CKAN GovData Catalog


Ziel ist es über den SPARQL-Endpoint den gesamten GovData Katalog mit allen für die stratifizierte Zufallsstichprobe relevanten Feldern zu fetchen.
Aus dem Katalog werden im Anschluss Strata abgeleitet und der Katalog unterteilt.

Aus diesen Stata werden im Anschluss Stichproben gezogen um die Gesamtheit des Katalogs möglichst akkurat durch die Stichprobe abzubilden.

Im ersten Schritt wird dabei versucht über pagination der gesamte GovData Katalog über Sparql-Queries zu fetchen

In [7]:
# 13. Mai 2026 09:06:10 German Time UTC+2


from datetime import datetime

import requests
import json
from pathlib import Path

BASE_URL = "https://www.govdata.de/ckan/api/3/action/package_search"

out = Path("data/raw")
out.mkdir(parents=True, exist_ok=True)

rows = 1000
start = 0

with open(out / f"govdata_ckan_snapshot{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jsonl", "w", encoding="utf-8") as f:
    while True:
        print(f"Fetching packages {start} to {start + rows}...")
        params = {"rows": rows, "start": start}
        r = requests.get(BASE_URL, params=params, timeout=60)
        r.raise_for_status()

        data = r.json()["result"]
        packages = data["results"]
        total = data["count"]

        for package in packages:
            f.write(json.dumps(package, ensure_ascii=False) + "\n")

        print(f"{start + len(packages)} / {total}")

        if start + rows >= total:
            break

        start += rows

Fetching packages 0 to 1000...
1000 / 151579
Fetching packages 1000 to 2000...
2000 / 151579
Fetching packages 2000 to 3000...
3000 / 151579
Fetching packages 3000 to 4000...
4000 / 151579
Fetching packages 4000 to 5000...
5000 / 151579
Fetching packages 5000 to 6000...
6000 / 151579
Fetching packages 6000 to 7000...
7000 / 151579
Fetching packages 7000 to 8000...
8000 / 151579
Fetching packages 8000 to 9000...
9000 / 151579
Fetching packages 9000 to 10000...
10000 / 151579
Fetching packages 10000 to 11000...
11000 / 151579
Fetching packages 11000 to 12000...
12000 / 151579
Fetching packages 12000 to 13000...
13000 / 151579
Fetching packages 13000 to 14000...
14000 / 151579
Fetching packages 14000 to 15000...
15000 / 151579
Fetching packages 15000 to 16000...
16000 / 151579
Fetching packages 16000 to 17000...
17000 / 151579
Fetching packages 17000 to 18000...
18000 / 151579
Fetching packages 18000 to 19000...
19000 / 151579
Fetching packages 19000 to 20000...
20000 / 151579
Fetching pa

Im nächsten Schritt laden wir die Daten in ein Dataframe und checken die Daten auf sinnvolle Kategorien um Stata zu bilden.

In [8]:
import json
from pathlib import Path

import pandas as pd

snapshot_path = Path("data/raw/govdata_ckan_snapshot2026-05-15_09-57-31.jsonl")
if not snapshot_path.exists():
    raise FileNotFoundError(
        f"{snapshot_path} nicht gefunden. Bitte zuerst die Fetch-Zelle ausführen."
    )

records = [
    json.loads(line)
    for line in snapshot_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

df = pd.json_normalize(records)

print(f"Datensätze: {len(df):,}")
print(f"Spalten: {len(df.columns):,}")
print()


def _normalize_item(item):
    if isinstance(item, dict):
        for key in ("name", "title", "display_name", "label", "id", "type"):
            value = item.get(key)
            if value not in (None, ""):
                return value
        return str(item)
    return item


def _top_values(series, label, n=5):
    values = []
    for value in series.dropna():
        if isinstance(value, list):
            values.extend(_normalize_item(item) for item in value)
        else:
            values.append(_normalize_item(value))

    values = pd.Series([value for value in values if value not in (None, "")])
    if values.empty:
        print(f"{label}: keine Werte gefunden")
        print()
        return

    print(f"Top {n} {label}:")
    print(values.astype(str).value_counts().head(n).to_string())
    print()


print("Wichtige Felder im Überblick")
print("-" * 32)

for column_name, label in [
    ("organization.name", "Organisationen"),
    ("license_id", "Lizenzen"),
    ("groups", "Themen / Groups"),
    ("tags", "Tags"),
]:
    if column_name in df.columns:
        _top_values(df[column_name], label)

resource_records = []
for record in records:
    for resource in record.get("resources", []):
        resource_records.append(resource)

resource_df = pd.DataFrame(resource_records)
if not resource_df.empty:
    print(f"Ressourcen: {len(resource_df):,}")
    print()
    for column_name, label in [
        ("format", "Ressourcen-Formate"),
        ("mimetype", "MIME-Typen"),
        ("resource_type", "Ressourcen-Typen"),
        ("type", "Ressourcen-Klassen"),
    ]:
        if column_name in resource_df.columns:
            _top_values(resource_df[column_name], label)


Datensätze: 151,579
Spalten: 39

Wichtige Felder im Überblick
--------------------------------
Top 5 Organisationen:
land-schleswig-holstein      28336
gdi-de                       27181
open-data-bayern             24687
open-data-brandenburg        17908
transparenzportal-hamburg    13940

Top 5 Lizenzen:
http://dcat-ap.de/def/licenses/dl-zero-de/2.0    534
http://dcat-ap.de/def/licenses/dl-by-de/2.0      305
http://dcat-ap.de/def/licenses/dl-by-de/2_0      275
http://dcat-ap.de/def/licenses/dl-zero-de/2_0     18
http://dcat-ap.de/def/licenses/official-work      14

Top 5 Themen / Groups:
regi    46578
envi    44443
gove    41971
econ    24510
soci    23060

Top 5 Tags:
opendata                52026
inspireidentifiziert    19396
b-plan                  19356
bauleitplanung          13411
vermessung               9450

Ressourcen: 560,381

Top 5 Ressourcen-Formate:
http://publications.europa.eu/resource/authority/file-type/NETCDF    127957
http://publications.europa.eu/resource/author

#### Erste Erkenntnisse aus dem Dataframe

Es gibt unglaublich viele Distributionen mit dem NETCDF File Format. NETCDF ist ein Binär-Format und auf den ersten Blick ist es unmöglich von so einem Format die Qualität zu messen.
Eine andere sehr interessante Erkenntnis ist die Menge an Distributionen für Datensätze die mindestens eine Distribution mit dem NETCDF File Format haben.

Es wurde am 15.05.2026 folgende SPARQL-Query gegen den Triple-Store von govdata.de ausgeführt:

```sql
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dct:  <http://purl.org/dc/terms/>

SELECT (AVG(?distributionCount) AS ?averageDistributionsPerDataset)
WHERE {
  {
    SELECT ?dataset (COUNT(DISTINCT ?distribution) AS ?distributionCount)
    WHERE {
      ?dataset a dcat:Dataset ;
               dcat:distribution ?distribution .

      FILTER EXISTS {
        ?dataset dcat:distribution ?netcdfDistribution .
        ?netcdfDistribution dct:format <http://publications.europa.eu/resource/authority/file-type/NETCDF> .
      }
    }
    GROUP BY ?dataset
  }
}
```

Mit folgendem Ergebnis:

```json
{ "head": {
    "vars": [ "averageDistributionsPerDataset" ]
  } ,
  "results": {
    "bindings": [
      { 
        "averageDistributionsPerDataset": { "type": "literal" , "datatype": "http://www.w3.org/2001/XMLSchema#decimal" , "value": "109.962888665997993981945838" }
      }
    ]
  }
}
```

Das sind IM SCHNITT knapp 110 Distributionen für ingesamt ca 1600 Datensätze die dieses Fileformat enthalten.
Wie soll so etwas bewertet werden??? 

Zur Abgrenzung der Datenmenge und zur Simplifizierung des Versuchsaufbaus werden diese 1600 Datensätze für die statifizierte Stichprobe ignoriert.
Da ein wichtiger Punkt die Weiternutzung der Daten ist und damit auch die eigentliche Ressource / Distribution eine große Rolle spielt, sollte diese zumindest Teilweise von messbarer Qualität sein.
Da NETCDF im Binärformat besteht ist es zumindest für das vorliegende Modell technisch nicht Aussagekräftig möglich die Qualität dieser Dateien zu überprüfen.

### Gedanken zur Abgrenzung der Datenmenge:

Es gibt insgesamt 1600 Datensätze mit NETCDF Distributionen. Es gibt insgesamt ca 160000 Datensätze im GovData Katalog. 
Mit einer Abgrenzung dieser Datensätze werden ca 1% der Datensätze ausgeschlossen. 

Das größte Problem was ich sehe ist die Tatsache, dass auf diesen 1600 Datensätzen im Schnitt 100 Distributionen liegen. Insgesamt sind es 127957 Distributionen die dieses Format haben.
Da diese Distributionen mit dem NETCDF Format binär sind, ist es technisch nicht möglich die Qualität dieser Distributionen zu messen. Es ist also nicht möglich die Qualität dieser Distributionen zu bewerten.

Aus diesem Grund werden diese Datensätze von der Stichprobe ausgeschlossen.

